
# Emotion Analysis with Pretrained RoBERTa

**Model:** `pysentimiento/robertuito-emotion-analysis`  
**Dataset:** goemotions_cleaned.csv

**Objective:** Evaluate a pretrained emotion classification model *without fine-tuning* using ranking-based and multi-label metrics, then fine-tune it and see the results


## 1. Imports and Setup
We begin by importing the required libraries for NLP inference, numerical computation, and data handling.


In [1]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import numpy as np


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



## 2. Load Pretrained Emotion Model
We load a RoBERTa-based emotion classifier from Hugging Face.  
This model is used *as-is* (no fine-tuning) to measure zero-shot performance.


In [2]:

tokenizer = AutoTokenizer.from_pretrained("pysentimiento/robertuito-emotion-analysis")
model = AutoModelForSequenceClassification.from_pretrained("pysentimiento/robertuito-emotion-analysis")

labels = model.config.id2label
labels


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2800.81it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-emotion-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{0: 'others',
 1: 'joy',
 2: 'sadness',
 3: 'anger',
 4: 'surprise',
 5: 'disgust',
 6: 'fear'}


## 3. Sanity Check: Single-Text Emotion Predictions
Before evaluating on a dataset, we test the model on a few example sentences and inspect the top-3 predicted emotions.


In [3]:

texts = [
    "I just got an amazing gift! I'm so happy!",
    "I'm feeling really sad today...",
    "This is so frustrating! I can't believe it.",
    "I love hanging out with my friends!"
]

for t in texts:
    inputs = tokenizer(t, return_tensors="pt")
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)

    top3 = torch.topk(probs, k=3)
    top3_probs = top3.values[0]
    top3_indices = top3.indices[0]
    top3_emotions = [labels[i.item()] for i in top3_indices]

    print(f"Text: {t}")
    for i in range(3):
        print(f"  {i+1}. {top3_emotions[i]} ({top3_probs[i].item()*100:.2f}%)")
    print()


Text: I just got an amazing gift! I'm so happy!
  1. joy (98.48%)
  2. surprise (0.64%)
  3. fear (0.24%)

Text: I'm feeling really sad today...
  1. sadness (99.29%)
  2. others (0.21%)
  3. fear (0.17%)

Text: This is so frustrating! I can't believe it.
  1. sadness (60.58%)
  2. fear (24.44%)
  3. surprise (5.21%)

Text: I love hanging out with my friends!
  1. joy (98.15%)
  2. others (0.68%)
  3. disgust (0.30%)




## 4. Load GoEmotions Dataset
We load the GoEmotions dataset and extract:
- Text data
- Multi-label emotion annotations


In [4]:

df = pd.read_csv("goemotions_1.csv")

emotion_cols = df.columns[9:]
df = df[["text"] + list(emotion_cols)]

df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'goemotions_1.csv'


## 5. Batch Inference (Efficient Evaluation)
To improve efficiency, predictions are generated in batches rather than one text at a time.


In [ ]:

batch_size = 32
predictions = []

for i in range(0, len(df), batch_size):
    batch_texts = df["text"].iloc[i:i+batch_size].tolist()
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    predictions.extend(probs.detach().numpy())

predictions = np.array(predictions)



## 6. Ground Truth Labels
The dataset contains multi-hot encoded emotion labels.


In [ ]:

true_labels = df[emotion_cols].astype(int).values
true_labels.shape



## 7. Evaluation Metric: Top-k Accuracy (k = 3)

**Definition:**  
A prediction is counted as correct if *at least one* of the top-k predicted emotions matches any ground-truth label.

This metric is suitable for:
- Multi-label emotion classification
- Ambiguous or overlapping emotions
- Zero-shot evaluation


In [ ]:

top_k = 3
correct_count = 0

for i in range(len(true_labels)):
    topk_preds = np.argsort(predictions[i])[::-1][:top_k]
    if any(true_labels[i][idx] == 1 for idx in topk_preds):
        correct_count += 1

top3_accuracy = correct_count / len(true_labels)
print(f"Top-{top_k} accuracy: {top3_accuracy*100:.2f}%")



## 8. Summary and Next Steps

**What this notebook shows:**
- Zero-shot emotion detection works reasonably well
- Top-k accuracy captures partial correctness in subjective labels

**Next steps:**
- Introduce probability thresholds
- Compute Micro-F1 and Jaccard similarity
- Fine-tune the model and compare against this baseline
